In [5]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

In [6]:
# Reading data from web
myretaildata = pd.read_excel('http://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx')

In [7]:
myretaildata.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [8]:
# Data cleaning
myretaildata['Description']=myretaildata['Description'].str.strip() # removes spces from the leading whitespace
myretaildata.dropna(axis=0,subset=['InvoiceNo'],inplace=True) #removes duplicate invoice
myretaildata['InvoiceNo'] = myretaildata['InvoiceNo'].astype('str') #converting the invoice number to string
myretaildata = myretaildata[~myretaildata['InvoiceNo'].str.contains('C')] #removes the credit transaction
myretaildata.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [9]:
myretaildata['Country'].value_counts()

Country
United Kingdom          487622
Germany                   9042
France                    8408
EIRE                      7894
Spain                     2485
Netherlands               2363
Belgium                   2031
Switzerland               1967
Portugal                  1501
Australia                 1185
Norway                    1072
Italy                      758
Channel Islands            748
Finland                    685
Cyprus                     614
Sweden                     451
Unspecified                446
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     295
Hong Kong                  284
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         58


In [10]:
# separating transactions for Germany
mybasket = (myretaildata[myretaildata['Country']=="Germany"]
            .groupby(['InvoiceNo','Description'])['Quantity']
            .sum().unstack().reset_index().fillna(0)
            .set_index('InvoiceNo'))

In [11]:
mybasket.head()

Description,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE SKULLS,...,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
InvoiceNo,,,,,,,,,,,,,,,,,,,,,
536527,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536840,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536861,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536967,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536983,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# converting all positive values to 1 and everything else to 0
def my_encode_unit(x):
    if x<=0:
        return 0
    if x>=1:
        return 1
my_basket_sets = mybasket.apply(lambda x: x.map(lambda y: 1 if y >= 1 else 0))
# my_basket_sets = mybasket.applymap(my_encode_unit)
my_basket_sets.drop('POSTAGE',inplace=True,axis=1) #remove 'postage' as an item

In [15]:
# Training Model
# Generating frequent items
my_frequent_items = apriori(my_basket_sets,min_support=0.07,use_colnames=True)

c:\Users\DELL\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [16]:
# generating rules
my_rules = association_rules(my_frequent_items,metric='lift',min_threshold=1)

In [19]:
# viewing top 100 rules
my_rules.head(100)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),0.137856,0.245077,0.074398,0.539683,2.202098,1.0,0.040613,1.640006,0.633174,0.241135,0.390246,0.421627
1,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),frozenset({PLASTERS IN TIN WOODLAND ANIMALS}),0.245077,0.137856,0.074398,0.303571,2.202098,1.0,0.040613,1.237951,0.723103,0.241135,0.192214,0.421627
2,frozenset({ROUND SNACK BOXES SET OF 4 FRUITS}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),0.157549,0.245077,0.131291,0.833333,3.400298,1.0,0.092679,4.529540,0.837922,0.483871,0.779227,0.684524
3,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),frozenset({ROUND SNACK BOXES SET OF 4 FRUITS}),0.245077,0.157549,0.131291,0.535714,3.400298,1.0,0.092679,1.814509,0.935072,0.483871,0.448887,0.684524
4,frozenset({SPACEBOY LUNCH BOX}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),0.102845,0.245077,0.070022,0.680851,2.778116,1.0,0.044817,2.365427,0.713415,0.251969,0.577243,0.483283
5,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),frozenset({SPACEBOY LUNCH BOX}),0.245077,0.102845,0.070022,0.285714,2.778116,1.0,0.044817,1.256018,0.847826,0.251969,0.203833,0.483283


In [21]:
my_basket_sets['ROUND SNACK BOXES SET OF4 WOODLAND'].sum()

np.int64(112)

In [22]:
my_basket_sets['SPACEBOY LUNCH BOX'].sum()

np.int64(47)

In [23]:
# filtering rules based on condition
my_rules[ (my_rules['lift']>=3) &
       (my_rules['confidence']>=0.3)]

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
2,frozenset({ROUND SNACK BOXES SET OF 4 FRUITS}),frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),0.157549,0.245077,0.131291,0.833333,3.400298,1.0,0.092679,4.529540,0.837922,0.483871,0.779227,0.684524
3,frozenset({ROUND SNACK BOXES SET OF4 WOODLAND}),frozenset({ROUND SNACK BOXES SET OF 4 FRUITS}),0.245077,0.157549,0.131291,0.535714,3.400298,1.0,0.092679,1.814509,0.935072,0.483871,0.448887,0.684524
